# 甄数：Colab 摸底

目的：测出 Robometer 和 TOPReward 在 Colab 上的真实接口、速度和显存，好写适配器。大约 15 到 30 分钟。

**怎么跑**
1. 菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**（有 Pro 就选 L4 或 A100）。
2. 左边「文件」面板，把本机的 `dist/assay-0.1.0-py3-none-any.whl` 拖进 `/content/`。
3. 菜单「代码执行程序 → 全部运行」。有一格报错没关系，后面照常跑，报错本身也是结论。如果又弹出「session crashed … RAM」，记下是第几格，直接把笔记本下载回来就行。
4. 跑完后「文件 → 下载 → 下载 .ipynb」，把文件放到项目的 `notebooks/colab_probe.out.ipynb`，告诉 Claude。

模型权重来自 Hugging Face 公开仓库（Robometer-4B 约 9 GB，Qwen3-VL-8B 约 16 GB），不需要登录。


In [1]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [2]:
# 2. 装依赖
!pip install -q "lerobot[topreward]" 2>&1 | tail -3
!pip install -q "lerobot[robometer]" 2>&1 | tail -3
!pip install -q /content/assay-0.1.0-py3-none-any.whl 2>&1 | tail -3
import importlib.metadata as md
for p in ["lerobot", "torch", "transformers", "assay", "av"]:
    try:
        print(p, md.version(p))
    except Exception as e:
        print(p, "没装上", e)

lerobot 0.6.1
torch 2.11.0+cu130
transformers 5.5.4
assay 0.1.0
av 19.0.0


In [3]:
# 3. 从 Zenodo 按需取 RoboVAD：标注、关节数据，加 3 条示范的左路视频
import csv, io, json, traceback, zipfile
PICK = []
try:
    from assay.eval.robovad_fetch import HttpRangeFile, ROBOVAD_URL, SMALL_PREFIXES, extract
    z = zipfile.ZipFile(io.BufferedReader(HttpRangeFile(ROBOVAD_URL), buffer_size=1 << 20))
    extract(z, "/content/robovad", SMALL_PREFIXES)
    ann = json.load(open("/content/robovad/annotations.json"))
    test = [r["episode_name"] for r in csv.DictReader(open("/content/robovad/splits/test.csv"))]
    normal = next(n for n in test if n.startswith("h_pickandplace_ep") and n not in ann)
    rings = next(n for n in test if n.startswith("h_rings_on_peg") and n in ann)
    PICK = ["h_pickandplace_ep9", normal, rings]
    extract(z, "/content/robovad", tuple(f"RoboVAD/videos/{n}_left.mp4" for n in PICK))
    for n in PICK:
        print(n, ann.get(n, "正常"))
except Exception:
    traceback.print_exc()

h_pickandplace_ep9 [{'start_frame': 118, 'end_frame': 135, 'type': 'object_drop', 'categories': {'severity': 'major', 'visibility': 'visible_in_both'}}, {'start_frame': 213, 'end_frame': 233, 'type': 'collision', 'categories': {'severity': 'major', 'visibility': 'visible_in_both'}}]
h_pickandplace_ep16 正常
h_rings_on_peg_ep1 [{'start_frame': 129, 'end_frame': 148, 'type': 'failed_grasp', 'categories': {'severity': 'major', 'visibility': 'visible_in_both'}}, {'start_frame': 220, 'end_frame': 228, 'type': 'failed_grasp', 'categories': {'severity': 'major', 'visibility': 'visible_in_both'}}, {'start_frame': 288, 'end_frame': 290, 'type': 'failed_grasp', 'categories': {'severity': 'major', 'visibility': 'visible_in_both'}}]


In [4]:
# 4. 读帧：每秒 2 帧，长边 448
import time, traceback
CLIPS = {}
try:
    from assay.ingest.robovad import read_robovad
    from assay.video import read_clip
    eps = {e.id: e for e in read_robovad("/content/robovad", names=PICK).episodes}
    for n in PICK:
        t = time.time()
        frames, times = read_clip(eps[n].cameras["left"], sample_fps=2.0, max_side=448)
        CLIPS[n] = (frames, times, eps[n].task)
        print(n, frames.shape, f"读帧 {time.time() - t:.1f}s", eps[n].task)
except Exception:
    traceback.print_exc()

h_pickandplace_ep9 (26, 252, 448, 3) 读帧 1.9s Pick up the cube and place it in the container.
h_pickandplace_ep16 (18, 252, 448, 3) 读帧 1.2s Pick up the cube and place it in the container.
h_rings_on_peg_ep1 (43, 252, 448, 3) 读帧 3.7s Put the ring onto the peg.


In [5]:
# 5a. 先把 Robometer 权重下载到磁盘（关掉 hf_xet，避免下载时吃内存），看剩余内存
import os, shutil, traceback
os.environ["HF_HUB_DISABLE_XET"] = "1"
!pip install -q psutil
import psutil
def ram():
    m = psutil.virtual_memory()
    return f"内存 可用 {m.available / 1e9:.1f} / 共 {m.total / 1e9:.1f} GB"
try:
    from huggingface_hub import snapshot_download
    t = time.time()
    path = snapshot_download("lerobot/Robometer-4B")
    print("下载到", path, f"{time.time() - t:.0f}s")
    print(ram(), "；磁盘剩余", shutil.disk_usage("/content").free // 10**9, "GB")
except Exception:
    traceback.print_exc()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

下载到 /root/.cache/huggingface/hub/models--lerobot--Robometer-4B/snapshots/db167a7c369a3ee59cda801fe33ca9da560b1662 1s
内存 可用 12.2 / 共 13.6 GB ；磁盘剩余 48 GB


In [6]:
# 5b. Robometer：看加载函数接受哪些参数，按 FP16、低内存的方式加载；看接口源码
import dataclasses, inspect, traceback, torch
rm = None
try:
    from lerobot.rewards.robometer import RobometerConfig, RobometerRewardModel
    from lerobot.rewards.robometer.modeling_robometer import ROBOMETER_FEATURE_PREFIX
    from lerobot.rewards.robometer.processor_robometer import RobometerEncoderProcessorStep
    print("RobometerConfig 字段", [f.name for f in dataclasses.fields(RobometerConfig)])
    print("from_pretrained 签名", inspect.signature(RobometerRewardModel.from_pretrained))
    rcfg = RobometerConfig(pretrained_path="lerobot/Robometer-4B", device="cuda", reward_output="progress")
    for f in ("dtype", "torch_dtype"):
        if hasattr(rcfg, f):
            setattr(rcfg, f, "float16")
            print("配置里设了", f, "= float16")
    print(rcfg)
    print("加载前", ram())
    for kw in [dict(torch_dtype=torch.float16, low_cpu_mem_usage=True), dict(dtype=torch.float16), {}]:
        try:
            torch.cuda.reset_peak_memory_stats()
            t = time.time()
            rm = RobometerRewardModel.from_pretrained(rcfg.pretrained_path, config=rcfg, **kw)
            print("加载参数", kw, f"用了 {time.time() - t:.0f}s")
            break
        except TypeError as e:
            print("不接受", kw, "：", repr(e)[:300])
    print("加载后", ram(), f"；显存 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print("参数 dtype", next(rm.parameters()).dtype, "设备", next(rm.parameters()).device)
    if torch.cuda.get_device_capability()[0] < 8 and next(rm.parameters()).dtype == torch.bfloat16:
        rm = rm.to(torch.float16)
        print("这块卡不支持 BF16，转成 FP16")
    print("forward 签名", inspect.signature(rm.forward))
    print("==== compute_reward 源码 ====")
    print(inspect.getsource(rm.compute_reward)[:6000])
except Exception:
    traceback.print_exc()
    try:
        print("==== from_pretrained 源码（加载失败时看怎么改）====")
        print(inspect.getsource(RobometerRewardModel.from_pretrained)[:5000])
    except Exception:
        pass

RobometerConfig 字段 ['input_features', 'output_features', 'device', 'pretrained_path', 'pretrained_revision', 'push_to_hub', 'repo_id', 'license', 'tags', 'private', 'image_key', 'task_key', 'default_task', 'max_frames', 'reward_output', 'success_threshold', 'base_model_id', 'torch_dtype', 'use_multi_image', 'use_per_frame_progress_token', 'average_temporal_patches', 'frame_pooling', 'frame_pooling_attn_temperature', 'progress_loss_type', 'progress_discrete_bins', 'vlm_config', 'normalization_mapping']
from_pretrained 签名 (pretrained_name_or_path: str | pathlib._local.Path, *, config: lerobot.configs.rewards.RewardModelConfig | None = None, force_download: bool = False, resume_download: bool | None = None, proxies: dict | None = None, token: str | bool | None = None, cache_dir: str | pathlib._local.Path | None = None, local_files_only: bool = False, revision: str | None = None, strict: bool = False, **kwargs) -> ~T


配置里设了 torch_dtype = float16
RobometerConfig(input_features={'observation.images.top': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224))}, output_features={'progress': PolicyFeature(type=<FeatureType.REWARD: 'REWARD'>, shape=(1,)), 'success': PolicyFeature(type=<FeatureType.REWARD: 'REWARD'>, shape=(1,))}, device='cuda', pretrained_path='lerobot/Robometer-4B', pretrained_revision=None, push_to_hub=False, repo_id=None, license='apache-2.0', tags=['reward-model', 'vision-language', 'qwen3-vl', 'zero-shot'], private=None, image_key='observation.images.top', task_key='task', default_task=None, max_frames=8, reward_output='progress', success_threshold=0.5, base_model_id='Qwen/Qwen3-VL-4B-Instruct', torch_dtype='float16', use_multi_image=True, use_per_frame_progress_token=True, average_temporal_patches=True, frame_pooling='mean', frame_pooling_attn_temperature=1.0, progress_loss_type='discrete', progress_discrete_bins=10, vlm_config={'transformers_version': '5.5.4', 'arc

In [7]:
# 6. Robometer：整段视频调一次，看返回值、耗时、显存
import traceback


def to_cuda(batch):
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in batch.items()}


try:
    enc = RobometerEncoderProcessorStep(
        base_model_id=rcfg.base_model_id,
        use_multi_image=rcfg.use_multi_image,
        use_per_frame_progress_token=rcfg.use_per_frame_progress_token,
        max_frames=rcfg.max_frames,
    )
    for n in PICK:
        frames, times, task = CLIPS[n]
        encoded = enc.encode_samples([(frames, task)])
        batch = to_cuda({f"{ROBOMETER_FEATURE_PREFIX}{k}": v for k, v in encoded.items()})
        if n == PICK[0]:
            print("batch 里的键和形状", {k: getattr(v, "shape", type(v)) for k, v in batch.items()})
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        with torch.no_grad():
            r = rm.compute_reward(batch)
        print(n, "返回", type(r), getattr(r, "shape", None), r, f"{time.time() - t:.1f}s",
              f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
except Exception:
    traceback.print_exc()

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2341: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


batch 里的键和形状 {'observation.robometer.input_ids': torch.Size([1, 3225]), 'observation.robometer.attention_mask': torch.Size([1, 3225]), 'observation.robometer.mm_token_type_ids': torch.Size([1, 3225]), 'observation.robometer.pixel_values': torch.Size([11648, 1536]), 'observation.robometer.image_grid_thw': torch.Size([26, 3]), 'observation.robometer.prog_token_id': <class 'int'>, 'observation.robometer.vision_start_token_id': <class 'int'>, 'observation.robometer.vision_end_token_id': <class 'int'>, 'observation.robometer.video_merge_size': <class 'int'>}


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


h_pickandplace_ep9 返回 <class 'torch.Tensor'> torch.Size([1]) tensor([0.8606], device='cuda:0') 8.1s 峰值 13.5 GB


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


h_pickandplace_ep16 返回 <class 'torch.Tensor'> torch.Size([1]) tensor([0.9565], device='cuda:0') 3.3s 峰值 11.5 GB


Traceback (most recent call last):
  File "/tmp/ipykernel_12881/3175658348.py", line 25, in <cell line: 0>
    r = rm.compute_reward(batch)
  File "/usr/local/lib/python3.13/dist-packages/lerobot/rewards/robometer/modeling_robometer.py", line 267, in compute_reward
    progress_logits, success_logits = self._compute_rbm_logits(inputs)
                                      ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/lerobot/rewards/robometer/modeling_robometer.py", line 313, in _compute_rbm_logits
    outputs = self.model(**inputs, output_hidden_states=True, return_dict=True)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-p

In [8]:
# 7. Robometer：模型内部有没有逐帧进度（看 forward 的输出和建模源码）
import traceback
try:
    with torch.no_grad():
        out = rm.forward(batch)
    print(type(out))
    if isinstance(out, dict):
        for k, v in out.items():
            print(k, getattr(v, "shape", type(v)))
    elif isinstance(out, (tuple, list)):
        for i, v in enumerate(out):
            print(i, getattr(v, "shape", type(v)))
    else:
        print(out)
except Exception:
    traceback.print_exc()
try:
    import lerobot.rewards.robometer.modeling_robometer as mr
    src = inspect.getsource(mr)
    i = src.find("progress")
    print("==== modeling_robometer.py 里提到 progress 的附近 ====")
    print(src[max(0, i - 2000): i + 4000])
except Exception:
    traceback.print_exc()

==== modeling_robometer.py 里提到 progress 的附近 ====
# Copyright 2026 Anthony Liang, Yigit Korkmaz, Stephen Tu, Erdem Bıyık, Jesse Zhang
# and The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""ROBOMETER: Scaling General-Purpose Robotic Reward Models via Trajectory Comparisons.

Paper:         https://arxiv.org/abs/2603.02115
Project:       https://robometer.github.io
Original code: https://github.com/aliang8/robometer
Model:         https://huggingface

Traceback (most recent call last):
  File "/tmp/ipykernel_12881/4185221633.py", line 5, in <cell line: 0>
    out = rm.forward(batch)
  File "/usr/local/lib/python3.13/dist-packages/lerobot/rewards/pretrained.py", line 165, in forward
    raise NotImplementedError(
        f"{self.__class__.__name__} is not trainable. Only use compute_reward() for inference."
    )
NotImplementedError: RobometerRewardModel is not trainable. Only use compute_reward() for inference.


In [9]:
# 8. Robometer：前 k 帧逐段调用 8 次，拼进度曲线，记总耗时
import numpy as np, traceback
try:
    for n in PICK:
        frames, times, task = CLIPS[n]
        ks = np.unique(np.linspace(2, len(frames), 8).astype(int))
        t = time.time()
        prog = []
        for k in ks:
            encoded = enc.encode_samples([(frames[:k], task)])
            b = to_cuda({f"{ROBOMETER_FEATURE_PREFIX}{kk}": v for kk, v in encoded.items()})
            with torch.no_grad():
                prog.append(float(torch.as_tensor(rm.compute_reward(b)).flatten()[0]))
        print(n, "锚点秒数", [round(float(times[k - 1]), 1) for k in ks],
              "进度", [round(p, 2) for p in prog], f"共 {time.time() - t:.1f}s")
except Exception:
    traceback.print_exc()

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


h_pickandplace_ep9 锚点秒数 [0.5, 2.0, 3.5, 5.5, 7.0, 9.0, 10.5, 12.5] 进度 [0.16, 0.36, 0.71, 0.33, 0.78, 0.81, 0.78, 0.86] 共 22.0s


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


h_pickandplace_ep16 锚点秒数 [0.5, 1.5, 2.5, 3.5, 5.0, 6.0, 7.0, 8.5] 进度 [0.21, 0.28, 0.27, 0.48, 0.77, 0.83, 0.91, 0.96] 共 14.0s


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Traceback (most recent call last):
  File "/tmp/ipykernel_12881/2445486037.py", line 13, in <cell line: 0>
    prog.append(float(torch.as_tensor(rm.compute_reward(b)).flatten()[0]))
                                      ~~~~~~~~~~~~~~~~~^^^
  File "/usr/local/lib/python3.13/dist-packages/lerobot/rewards/robometer/modeling_robometer.py", line 267, in compute_reward
    progress_logits, success_logits = self._compute_rbm_logits(inputs)
                                      ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/usr/loc

In [10]:
# 9. 腾显存给 TOPReward
import gc
try:
    del rm
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"现在占用 {torch.cuda.memory_allocated() / 1e9:.1f} GB")

现在占用 0.2 GB


In [ ]:
# 10. TOPReward：先试 8B，装不下换 4B；看接口源码
import traceback
TOP = None
try:
    from lerobot.rewards import make_reward_model, make_reward_model_config, make_reward_pre_post_processors
    for vlm in ["Qwen/Qwen3-VL-8B-Instruct", "Qwen/Qwen3-VL-4B-Instruct"]:
        try:
            snapshot_download(vlm)  # 先下到磁盘
            print(vlm, "下载完", ram())
            torch.cuda.reset_peak_memory_stats()
            t = time.time()
            tcfg = make_reward_model_config("topreward", vlm_name=vlm, device="cuda", image_key="observation.images.top")
            for f in ("dtype", "torch_dtype"):
                if hasattr(tcfg, f):
                    setattr(tcfg, f, "float16")
            tm = make_reward_model(tcfg)
            tpre, tpost = make_reward_pre_post_processors(tcfg)
            TOP = vlm
            print(vlm, f"加载 {time.time() - t:.0f}s，显存 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB", ram())
            break
        except Exception as e:
            print(vlm, "加载失败：", repr(e)[:500])
            gc.collect()
            torch.cuda.empty_cache()
    print(tcfg)
    print("==== compute_reward 源码 ====")
    print(inspect.getsource(tm.compute_reward)[:6000])
    print("==== 预处理器 ====")
    print(tpre)
except Exception:
    traceback.print_exc()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Qwen/Qwen3-VL-8B-Instruct 下载完 内存 可用 9.2 / 共 13.6 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

In [ ]:
# 11. TOPReward：按预处理器的格式喂一段视频，看返回值、耗时
import traceback
try:
    frames, times, task = CLIPS[PICK[0]]
    video = torch.from_numpy(frames).permute(0, 3, 1, 2).float().div(255).unsqueeze(0)  # 1×T×C×H×W
    raw = {"observation.images.top": video, "task": [task]}
    try:
        b = tpre(raw)
    except Exception as e:
        print("直接过预处理器失败，打印预处理器各步源码：", repr(e)[:500])
        for step in getattr(tpre, "steps", []):
            print("----", type(step).__name__)
            print(inspect.getsource(type(step))[:3000])
        raise
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    with torch.no_grad():
        r = tm.compute_reward(b)
    print("返回", type(r), getattr(r, "shape", None), r, f"{time.time() - t:.1f}s",
          f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
except Exception:
    traceback.print_exc()

In [ ]:
# 12. 小结
print("GPU", torch.cuda.get_device_name(0), "；TOPReward 用的", TOP)
print("跑完了。菜单「文件 → 下载 → 下载 .ipynb」，放到项目 notebooks/colab_probe.out.ipynb。")